# Notebook 02: AgentCore Runtime Setup

## Learning Objectives
- Configure AgentCore Runtime for travel agent
- Create basic conversational agent using Strands
- Implement multi-turn dialogue patterns
- Test conversation flow and state management

## Prerequisites
- Completed Notebook 01 (Foundation)
- AWS environment validated
- API keys configured

## Step 1: Connect to your AWS environment

In [1]:
import os

os.environ['AWS_REGION'] = 'us-east-1'

# APPROACH A: Use credentials
# os.environ['AWS_ACCESS_KEY_ID'] = 'your_access_key'
# os.environ['AWS_SECRET_ACCESS_KEY'] = 'your_secret_key'
# os.environ['AWS_SESSION_TOKEN'] = "your_session_token"

# APPROACH B: Use AWS SSO profile
#os.environ['AWS_PROFILE'] = 'your_profile'
# Remove any existing credential env vars to force profile usage
#for key in ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_SESSION_TOKEN']:
#    os.environ.pop(key, None)

os.environ["AWS_PROFILE"] = "agentcore-lab"
os.environ['AWS_REGION'] = 'us-east-1'

print("✅ AWS Profile set. Please restart kernel and run all cells.")

✅ AWS Profile set. Please restart kernel and run all cells.


In [2]:
import os
from dotenv import load_dotenv
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel

# Load environment variables
load_dotenv()

print("✅ AgentCore Runtime imports successful")

✅ AgentCore Runtime imports successful


## Step 2: Create Basic Travel Agent Tools

In [3]:
# Define basic travel tools for our agent
@tool
def get_travel_preferences():
    """Get user's travel preferences from memory"""
    # Mock implementation - will be enhanced in Memory notebook
    return {
        "hotel_type": "mid-range",
        "food_preference": "vegetarian",
        "budget_range": "moderate"
    }

@tool
def calculate_budget(total_budget: int, days: int):
    """Calculate daily budget allocation for travel"""
    # Basic budget breakdown
    daily_budget = total_budget / days
    allocation = {
        "flights": total_budget * 0.24,  # 24%
        "hotels": total_budget * 0.36,   # 36%
        "food": total_budget * 0.20,     # 20%
        "activities": total_budget * 0.16, # 16%
        "buffer": total_budget * 0.04     # 4%
    }
    return {
        "daily_budget": daily_budget,
        "allocation": allocation
    }

@tool
def get_destination_info(destination: str):
    """Get basic information about a travel destination"""
    # Mock implementation - will be enhanced with real APIs
    destinations = {
        "rome": {
            "country": "Italy",
            "currency": "EUR",
            "language": "Italian",
            "attractions": ["Colosseum", "Vatican", "Trevi Fountain"]
        },
        "florence": {
            "country": "Italy",
            "currency": "EUR",
            "language": "Italian",
            "attractions": ["Uffizi Gallery", "Ponte Vecchio", "Duomo"]
        },
        "venice": {
            "country": "Italy",
            "currency": "EUR",
            "language": "Italian",
            "attractions": ["St. Mark's Square", "Grand Canal", "Doge's Palace"]
        }
    }
    return destinations.get(destination.lower(), {"error": "Destination not found"})

print("✅ Travel agent tools defined")

✅ Travel agent tools defined


## Step 3: Create Travel Agent with Bedrock Model

In [10]:
# Initialize Bedrock model for the agent
model_id = "us.amazon.nova-2-lite-v1:0"
model = BedrockModel(model_id=model_id)

# Create the travel agent with system prompt
system_prompt = """
You are an AI Travel Companion specializing in planning trips to Italy. 
Your expertise includes:
- Flight and hotel recommendations
- Budget optimization and allocation
- Destination information and attractions
- Personalized recommendations based on user preferences

Always ask clarifying questions to better understand the user's needs.
Be helpful, friendly, and provide detailed explanations for your recommendations.
Remember user preferences and reference them in future interactions.
"""

travel_agent = Agent(
    model=model,
    tools=[get_travel_preferences, calculate_budget, get_destination_info],
    system_prompt=system_prompt
)

print("✅ Travel agent created with Bedrock model")

✅ Travel agent created with Bedrock model


## Step 4: Test Local Agent Functionality

In [11]:
# Test the agent locally before deploying to runtime
def test_travel_agent(user_input):
    """Test function for local agent invocation"""
    print(f"User: {user_input}")
    response = travel_agent(user_input)
    agent_response = response.message['content'][0]['text']
    print(f"Agent: {agent_response}")
    return agent_response

# Test basic functionality
print("🧪 Testing Travel Agent Locally")
print("=" * 50)

test_travel_agent("Hi, I want to plan a trip to Italy")

🧪 Testing Travel Agent Locally
User: Hi, I want to plan a trip to Italy
I'd be delighted to help you plan your trip to Italy! To create the perfect itinerary for you, I need to understand a few details about your preferences and plans.

Could you please share with me:

1. **Which cities or regions in Italy you're interested in visiting?** (e.g., Rome, Florence, Venice, Amalfi Coast, Tuscany, Sicily, etc.)

2. **How many days you plan to stay in Italy?**

3. **Your total budget for the trip?** (including flights, accommodation, food, activities, etc.)

4. **Travel preferences** such as:
   - Preferred travel style (luxury, budget-friendly, family-friendly, romantic, adventure, cultural)
   - Accommodation preferences (hotel, Airbnb, boutique, etc.)
   - Any specific interests (art, food, history, beaches, hiking, shopping, etc.)

Once I have this information, I can create a detailed plan for your Italian adventure!Agent: I'd be delighted to help you plan your trip to Italy! To create th

"I'd be delighted to help you plan your trip to Italy! To create the perfect itinerary for you, I need to understand a few details about your preferences and plans.\n\nCould you please share with me:\n\n1. **Which cities or regions in Italy you're interested in visiting?** (e.g., Rome, Florence, Venice, Amalfi Coast, Tuscany, Sicily, etc.)\n\n2. **How many days you plan to stay in Italy?**\n\n3. **Your total budget for the trip?** (including flights, accommodation, food, activities, etc.)\n\n4. **Travel preferences** such as:\n   - Preferred travel style (luxury, budget-friendly, family-friendly, romantic, adventure, cultural)\n   - Accommodation preferences (hotel, Airbnb, boutique, etc.)\n   - Any specific interests (art, food, history, beaches, hiking, shopping, etc.)\n\nOnce I have this information, I can create a detailed plan for your Italian adventure!"

In [12]:
# Test budget calculation
test_travel_agent("I have a budget of $5000 for a 10-day trip. How should I allocate it?")

User: I have a budget of $5000 for a 10-day trip. How should I allocate it?

Tool #1: calculate_budget
Perfect! Based on your $5,000 budget for a 10-day trip to Italy, here's how I'd recommend allocating your funds:

## Budget Allocation Breakdown:

**Daily Budget: $500 per day**

### Recommended Allocation:
- **Flights: $1,200** (24% of total budget) - This covers round-trip international flights from North America to major Italian airports (Rome, Milan, or Venice). Prices vary by origin, but this is a realistic estimate.
  
- **Accommodation: $1,800** (36% of total budget) - About $180 per night for quality hotels or nice Airbnb accommodations across 10 days. This allows for a mix of city-center hotels in major destinations and potentially nicer stays in scenic areas.

- **Food & Drinks: $1,000** (20% of total budget) - About $100 per day. This covers excellent meals including gelato, espresso, and local cuisine experiences. Italy offers great value for dining!

- **Activities & Attr

"Perfect! Based on your $5,000 budget for a 10-day trip to Italy, here's how I'd recommend allocating your funds:\n\n## Budget Allocation Breakdown:\n\n**Daily Budget: $500 per day**\n\n### Recommended Allocation:\n- **Flights: $1,200** (24% of total budget) - This covers round-trip international flights from North America to major Italian airports (Rome, Milan, or Venice). Prices vary by origin, but this is a realistic estimate.\n  \n- **Accommodation: $1,800** (36% of total budget) - About $180 per night for quality hotels or nice Airbnb accommodations across 10 days. This allows for a mix of city-center hotels in major destinations and potentially nicer stays in scenic areas.\n\n- **Food & Drinks: $1,000** (20% of total budget) - About $100 per day. This covers excellent meals including gelato, espresso, and local cuisine experiences. Italy offers great value for dining!\n\n- **Activities & Attractions: $800** (16% of total budget) - About $80 per day for museum entries, guided tour

In [13]:
# Test destination information
test_travel_agent("Tell me about Rome and what I should see there")

User: Tell me about Rome and what I should see there

Tool #2: get_destination_info

Tool #3: get_destination_info
# Exploring Rome: Italy's Eternal City

Rome is an incredible destination packed with history, art, and vibrant culture. As one of the world's most important historical cities, it offers a unique blend of ancient wonders, Renaissance masterpieces, and modern Italian life.

## Essential Attractions to See in Rome

### **Top Must-See Sites:**

**1. The Colosseum**
- **Why visit**: This iconic amphitheater is a symbol of ancient Roman engineering and gladiatorial contests
- **Tip**: Buy tickets in advance to avoid long lines and consider a guided tour for fascinating historical insights

**2. Vatican City & St. Peter's Basilica**
- **Why visit**: The spiritual center of Catholicism and home to Michelangelo's stunning Pietà
- **Must-see**: Vatican Museums and the Sistine Chapel (book tickets well in advance)
- **Tip**: Early morning or late afternoon visits offer better lighti

'# Exploring Rome: Italy\'s Eternal City\n\nRome is an incredible destination packed with history, art, and vibrant culture. As one of the world\'s most important historical cities, it offers a unique blend of ancient wonders, Renaissance masterpieces, and modern Italian life.\n\n## Essential Attractions to See in Rome\n\n### **Top Must-See Sites:**\n\n**1. The Colosseum**\n- **Why visit**: This iconic amphitheater is a symbol of ancient Roman engineering and gladiatorial contests\n- **Tip**: Buy tickets in advance to avoid long lines and consider a guided tour for fascinating historical insights\n\n**2. Vatican City & St. Peter\'s Basilica**\n- **Why visit**: The spiritual center of Catholicism and home to Michelangelo\'s stunning Pietà\n- **Must-see**: Vatican Museums and the Sistine Chapel (book tickets well in advance)\n- **Tip**: Early morning or late afternoon visits offer better lighting and fewer crowds\n\n**3. Trevi Fountain**\n- **Why visit**: The most beautiful Baroque fount

## Step 5: Prepare Agent for AgentCore Runtime

In [ ]:
%%writefile ../backend/runtime/simple_agent/travel_agent.py
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
import json

# Initialize AgentCore Runtime App
app = BedrockAgentCoreApp()

# Define travel tools
@tool
def get_travel_preferences():
    """Get user's travel preferences from memory"""
    return {
        "hotel_type": "mid-range",
        "food_preference": "vegetarian",
        "budget_range": "moderate"
    }

@tool
def calculate_budget(total_budget: int, days: int):
    """Calculate daily budget allocation for travel"""
    daily_budget = total_budget / days
    allocation = {
        "flights": total_budget * 0.24,
        "hotels": total_budget * 0.36,
        "food": total_budget * 0.20,
        "activities": total_budget * 0.16,
        "buffer": total_budget * 0.04
    }
    return {
        "daily_budget": daily_budget,
        "allocation": allocation
    }

@tool
def get_destination_info(destination: str):
    """Get basic information about a travel destination"""
    destinations = {
        "rome": {
            "country": "Italy",
            "currency": "EUR",
            "language": "Italian",
            "attractions": ["Colosseum", "Vatican", "Trevi Fountain"]
        },
        "florence": {
            "country": "Italy",
            "currency": "EUR",
            "language": "Italian",
            "attractions": ["Uffizi Gallery", "Ponte Vecchio", "Duomo"]
        },
        "venice": {
            "country": "Italy",
            "currency": "EUR",
            "language": "Italian",
            "attractions": ["St. Mark's Square", "Grand Canal", "Doge's Palace"]
        }
    }
    return destinations.get(destination.lower(), {"error": "Destination not found"})

# Initialize model and agent
model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
model = BedrockModel(model_id=model_id)

system_prompt = """
You are an AI Travel Companion specializing in planning trips to Italy. 
Your expertise includes:
- Flight and hotel recommendations
- Budget optimization and allocation
- Destination information and attractions
- Personalized recommendations based on user preferences

Always ask clarifying questions to better understand the user's needs.
Be helpful, friendly, and provide detailed explanations for your recommendations.
Remember user preferences and reference them in future interactions.
"""

travel_agent = Agent(
    model=model,
    tools=[get_travel_preferences, calculate_budget, get_destination_info],
    system_prompt=system_prompt
)

@app.entrypoint
def invoke_travel_agent(payload):
    """AgentCore Runtime entrypoint for travel agent"""
    user_input = payload.get("prompt", "")
    print(f"User input: {user_input}")
    
    response = travel_agent(user_input)
    agent_response = response.message['content'][0]['text']
    
    return agent_response

if __name__ == "__main__":
    app.run()

In [ ]:
%%writefile ../backend/runtime/simple_agent/requirements.txt
# Core Amazon Bedrock AgentCore dependencies
bedrock-agentcore>=1.0.5
bedrock-agentcore-starter-toolkit>=0.1.27

# AWS SDK and utilities
boto3>=1.40.62
python-dotenv>=1.2.1

# Agent framework
strands-agents>=1.14.0

## Step 6: Deploy to AgentCore Runtime

In [15]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import os
from pathlib import Path

# Initialize runtime deployment
boto_session = Session()
region = boto_session.region_name or "us-east-1"

# Change to project root for AgentCore configuration
original_dir = os.getcwd()
os.chdir('../backend/runtime/simple_agent')
project_root = os.getcwd()

agentcore_runtime = Runtime()
agent_name = "travel_companion_basic"

print(f"🚀 Configuring AgentCore Runtime deployment...")
print(f"Agent Name: {agent_name}")
print(f"Region: {region}")
print(f"Project Root: {project_root}")

# Configure with paths relative to project root
configure_response = agentcore_runtime.configure(
    entrypoint="travel_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    container_runtime=None  # Force CodeBuild for cross-platform build
)

print("✅ Runtime configuration complete")
print("💡 Using CodeBuild for cross-platform ARM64 deployment")
configure_response

Entrypoint parsed: file=/Users/bijonguha/BijonGuha/Drive/mastering-amazon-bedrock-agentcore/capstone_project/backend/runtime/simple_agent/travel_agent.py, bedrock_agentcore_name=travel_agent
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: travel_companion_basic


🚀 Configuring AgentCore Runtime deployment...
Agent Name: travel_companion_basic
Region: us-east-1
Project Root: /Users/bijonguha/BijonGuha/Drive/mastering-amazon-bedrock-agentcore/capstone_project/backend/runtime/simple_agent


💡 No container engine found (Docker/Finch/Podman not installed)

✓ Default deployment uses CodeBuild (no container engine needed), For local builds, install Docker, Finch, or 
Podman

Memory disabled
Network mode: PUBLIC


⚠️ Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64', so local builds
won't work.
Please use default launch command which will do a remote cross-platform build using code build.For deployment other
options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

📄 Generated Dockerfile: 
/Users/bijonguha/BijonGuha/Drive/mastering-amazon-bedrock-agentcore/capstone_project/backend/runtime/simple_agent/D
ockerfile

Generated .dockerignore: /Users/bijonguha/BijonGuha/Drive/mastering-amazon-bedrock-agentcore/capstone_project/backend/runtime/simple_agent/.dockerignore
Setting 'travel_companion_basic' as default agent
Bedrock AgentCore configured: /Users/bijonguha/BijonGuha/Drive/mastering-amazon-bedrock-agentcore/capstone_project/backend/runtime/simple_agent/.bedrock_agentcore.yaml


✅ Runtime configuration complete
💡 Using CodeBuild for cross-platform ARM64 deployment


ConfigureResult(config_path=PosixPath('/Users/bijonguha/BijonGuha/Drive/mastering-amazon-bedrock-agentcore/capstone_project/backend/runtime/simple_agent/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/bijonguha/BijonGuha/Drive/mastering-amazon-bedrock-agentcore/capstone_project/backend/runtime/simple_agent/Dockerfile'), dockerignore_path=PosixPath('/Users/bijonguha/BijonGuha/Drive/mastering-amazon-bedrock-agentcore/capstone_project/backend/runtime/simple_agent/.dockerignore'), runtime='None', runtime_type=None, region='us-east-1', account_id='016091700700', execution_role=None, ecr_repository=None, auto_create_ecr=True, s3_path=None, auto_create_s3=False, memory_id=None, network_mode='PUBLIC', network_subnets=None, network_security_groups=None, network_vpc_id=None)

In [16]:
# Launch the agent to AgentCore Runtime
print("🚀 Launching agent to AgentCore Runtime...")
print("This may take 5-10 minutes...")

launch_result = agentcore_runtime.launch()
print("✅ Launch initiated")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"ECR URI: {launch_result.ecr_uri}")

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'travel_companion_basic' to account 016091700700 (us-east-1)
Generated image tag: 20261004-150709-992
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: travel_companion_basic


🚀 Launching agent to AgentCore Runtime...
This may take 5-10 minutes...
Repository doesn't exist, creating new ECR repository: bedrock-agentcore-travel_companion_basic


ECR repository available: 016091700700.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-travel_companion_basic
Getting or creating execution role for agent: travel_companion_basic
Using AWS region: us-east-1, account ID: 016091700700
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-346339bc6e
Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-346339bc6e
Starting execution role creation process for agent: travel_companion_basic
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-us-east-1-346339bc6e
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-346339bc6e
✓ Role created: arn:aws:iam::016091700700:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-346339bc6e
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-travel_companion_basic
Role creation complete and ready for use with Bedrock AgentCore
Execution role available: arn:aws:iam::016091700700:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-346339bc6e
Preparing C

✅ Launch initiated
Agent ARN: arn:aws:bedrock-agentcore:us-east-1:016091700700:runtime/travel_companion_basic-SsLRKlGyGy
ECR URI: 016091700700.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-travel_companion_basic:20261004-150709-992


In [17]:
# Check deployment status
import time

print("⏳ Checking deployment status...")
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

while status not in end_status:
    print(f"Status: {status}")
    time.sleep(30)  # Check every 30 seconds
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']

print(f"\n🎉 Final Status: {status}")

if status == 'READY':
    print("✅ Agent successfully deployed to AgentCore Runtime!")
else:
    print("❌ Deployment failed. Check AWS console for details.")

# Return to original directory
os.chdir(original_dir)

⏳ Checking deployment status...


Retrieved Bedrock AgentCore status for: travel_companion_basic



🎉 Final Status: READY
✅ Agent successfully deployed to AgentCore Runtime!


## Step 7: Test Deployed Agent

In [18]:
# Test the deployed agent
if status == 'READY':
    print("🧪 Testing deployed Travel Agent")
    print("=" * 50)
    
    # Test basic interaction
    test_payload = {"prompt": "Hi, I want to plan a 10-day trip to Italy with a $5000 budget"}
    invoke_response = agentcore_runtime.invoke(test_payload)
    
    print(f"User: {test_payload['prompt']}")
    print(f"Agent: {invoke_response['response'][0]}")
else:
    print("⚠️ Cannot test - deployment not ready")

🧪 Testing deployed Travel Agent
User: Hi, I want to plan a 10-day trip to Italy with a $5000 budget
Agent: I'll help you plan a wonderful 10-day trip to Italy with your $5000 budget! Let me get some more information to personalize your travel experience.

To create the best itinerary for you, I need to understand:
- Do you have any preferred destinations or regions in Italy? (e.g., Rome, Florence, Amalfi Coast, Tuscany, Milan, etc.)
- Do you have any travel preferences? (e.g., food experiences, history & culture, beaches, shopping, luxury vs budget travel)
- Will you be traveling alone, with a partner, or in a group?
- Do you have any specific interests? (e.g., art, architecture, wine tasting, hiking, cooking classes)

Could you share these details with me so I can tailor your perfect Italian adventure?


In [19]:
# Test budget calculation functionality
if status == 'READY':
    test_payload = {"prompt": "Can you help me allocate my $5000 budget for 10 days?"}
    invoke_response = agentcore_runtime.invoke(test_payload)
    
    print(f"\nUser: {test_payload['prompt']}")
    print(f"Agent: {invoke_response['response'][0]}")


User: Can you help me allocate my $5000 budget for 10 days?
Agent: Perfect! Here's your budget allocation for your 10-day trip to Italy with a $5000 total budget:

## Daily Budget: $500 per day

### Budget Allocation Breakdown:
- **Flights**: $1,200 (24% of total budget)
- **Hotels**: $1,800 (36% of total budget) 
- **Food**: $1,000 (20% of total budget)
- **Activities & Experiences**: $800 (16% of total budget)
- **Buffer (for unexpected expenses)**: $200 (4% of total budget)

This allocation gives you a well-balanced travel experience. With $500 per day, you'll have plenty to cover all your essentials while still enjoying premium experiences in Italy.

Would you like me to:
1. Suggest specific destinations in Italy that would work well with this budget?
2. Help you plan an itinerary based on your interests?
3. Provide accommodation recommendations for specific cities?

Let me know what you'd like to explore next!


## Step 8: Multi-turn Conversation Test

In [20]:
# Test multi-turn conversation
if status == 'READY':
    print("🗣️ Testing Multi-turn Conversation")
    print("=" * 50)
    
    conversation = [
        "I want to visit Italy",
        "I prefer mid-range hotels and vegetarian food",
        "Tell me about Rome's attractions",
        "What about Florence?"
    ]
    
    for i, message in enumerate(conversation, 1):
        print(f"\n--- Turn {i} ---")
        test_payload = {"prompt": message}
        invoke_response = agentcore_runtime.invoke(test_payload)
        
        print(f"User: {message}")
        print(f"Agent: {invoke_response['response'][0][:200]}...")  # Truncate for readability

🗣️ Testing Multi-turn Conversation

--- Turn 1 ---
User: I want to visit Italy
Agent: "I apologize for the error. Let me get more specific information about popular destinations in Italy for you. Since \"Italy\" as a whole is too broad for the destination info tool, let me suggest some...

--- Turn 2 ---
User: I prefer mid-range hotels and vegetarian food
Agent: "Great preferences! With mid-range hotels and vegetarian food in mind, let me tailor some recommendations for your Italian trip.\n\nSince you're visiting Italy for 10 days with a $500/day budget, here...

--- Turn 3 ---
User: Tell me about Rome's attractions
Agent: "Perfect! Let me give you a comprehensive overview of Rome's amazing attractions that will work wonderfully for your 10-day Italian adventure.\n\n## Top Attractions in Rome (Perfect for your trip!)\n\...

--- Turn 4 ---
User: What about Florence?
Agent: "Excellent choice! Florence is one of Italy's most beautiful and culturally rich cities, perfect for your 10-day tr

## Step 9: Save Runtime Information

In [21]:
# Save runtime information for use in subsequent notebooks
import json

if status == 'READY':
    runtime_info = {
        "agent_name": agent_name,
        "agent_arn": launch_result.agent_arn,
        "agent_id": launch_result.agent_id,
        "ecr_uri": launch_result.ecr_uri,
        "region": region,
        "status": status
    }
    
    # Save to file for next notebooks
    with open('environments/runtime_info.json', 'w') as f:
        json.dump(runtime_info, f, indent=2)
    
    print("💾 Runtime information saved to environments/runtime_info.json")
    print("\n📋 Runtime Summary:")
    for key, value in runtime_info.items():
        print(f"  {key}: {value}")
else:
    print("⚠️ Runtime not ready - information not saved")

💾 Runtime information saved to environments/runtime_info.json

📋 Runtime Summary:
  agent_name: travel_companion_basic
  agent_arn: arn:aws:bedrock-agentcore:us-east-1:016091700700:runtime/travel_companion_basic-SsLRKlGyGy
  agent_id: travel_companion_basic-SsLRKlGyGy
  ecr_uri: 016091700700.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-travel_companion_basic:20261004-150709-992
  region: us-east-1
  status: READY


## Next Steps

✅ **Completed in this notebook:**
- AgentCore Runtime configuration and setup
- Basic travel agent with Strands and Bedrock
- Multi-turn conversation capabilities
- Production deployment to AWS

➡️ **Next: Notebook 03 - Gateway Integration**
- Integrate external APIs (flights, hotels, weather, currency)
- Create OpenAPI 3.0 specifications
- Set up MCP Gateway with OAuth
- Test real API integrations